In [ ]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

df_train=pd.read_csv("ClassAnnotations.csv")
df_test=pd.read_csv("TestClassAnnotations.csv")
df_val=pd.read_csv("ValClassAnnotations.csv")

train_abcd_values = df_train[['A', 'B', 'C', 'D']].values

abcd_means = train_abcd_values.mean(axis=0)
abcd_stds = train_abcd_values.std(axis=0)

abcd_stds = np.where(abcd_stds < 1e-6, 1e-6, abcd_stds)

print(f"Średnie (Train): {abcd_means}")
print(f"Odchylenia (Train): {abcd_stds}")

Średnie (Train): [0.1927001  4.75910318 0.19750723 0.68270492]
Odchylenia (Train): [ 0.10189307 12.65954761  0.0861146   0.05688187]


In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class_counts = df_train['label'].value_counts().sort_index().values
print(f"Liczebność klas w Train: {class_counts}")

class_weights = 1.0 / class_counts

sample_weights = [class_weights[label] for label in df_train['label']]
sampler_weights_tensor = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(
    weights=sampler_weights_tensor, 
    num_samples=len(sampler_weights_tensor), 
    replacement=True
)

Liczebność klas w Train: [ 418 1483  173]


In [ ]:
class MelanomaMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_dir, abcd_means, abcd_stds, transform=None):

        self.dataframe = dataframe.reset_index(drop=True) 
        self.img_dir = img_dir
        self.transform = transform
        
        self.abcd_means = abcd_means
        self.abcd_stds = abcd_stds

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_id = self.dataframe.loc[idx, 'image']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        raw_abcd = self.dataframe.loc[idx, ['A', 'B', 'C', 'D']].values.astype(np.float32)
        norm_abcd = (raw_abcd - self.abcd_means) / self.abcd_stds
        abcd_tensor = torch.tensor(norm_abcd, dtype=torch.float32)
        label = self.dataframe.loc[idx, 'label']
        label_tensor = torch.tensor(label, dtype=torch.long)

        return (image, abcd_tensor), label_tensor

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

train_folder="SegDataset/train/trainInput"
val_folder="SegDataset/val/valInput"

train_dataset = MelanomaMultiModalDataset(
    df_train, train_folder, abcd_means, abcd_stds, transform=train_transform)

val_dataset = MelanomaMultiModalDataset(
    df_val, val_folder, abcd_means, abcd_stds, transform=val_test_transform)

batch_size = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=train_sampler,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False
)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class MelanomaLateFusionModel(nn.Module):
    def __init__(self, num_classes=3, num_tabular_features=4):
        super(MelanomaLateFusionModel, self).__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.image_model = models.efficientnet_b0(weights=weights)
        self.image_model.classifier = nn.Identity()
        self.tabular_model = nn.Sequential(
            nn.Linear(num_tabular_features, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )
        fusion_size = 1280 + 16
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(fusion_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, tabular):
        img_features = self.image_model(image)
        tab_features = self.tabular_model(tabular)
        fused_features = torch.cat((img_features, tab_features), dim=1)
        output = self.classifier(fused_features)
        
        return output

#Sanity check
if __name__ == "__main__":
    model = MelanomaLateFusionModel(num_classes=3)
    dummy_images = torch.randn(8, 3, 224, 224)
    dummy_abcd = torch.randn(8, 4)
    
    predictions = model(dummy_images, dummy_abcd)
    
    print(f"Kształt wejścia obrazów: {dummy_images.shape}")
    print(f"Kształt wejścia ABCD: {dummy_abcd.shape}")
    print(f"Kształt wyjścia modelu: {predictions.shape}")
    print("Jeśli tu dotarłeś bez błędu, Twoja architektura działa perfekcyjnie!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/amay/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 23.1MB/s]


Kształt wejścia obrazów: torch.Size([8, 3, 224, 224])
Kształt wejścia ABCD: torch.Size([8, 4])
Kształt wyjścia modelu: torch.Size([8, 3])
Jeśli tu dotarłeś bez błędu, Twoja architektura działa perfekcyjnie!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Rozpoczynam trening na urządzeniu: {device}")

model = MelanomaLateFusionModel(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)

num_epochs = 20
best_val_loss = float('inf')
patience = 5 # Early Stopping
epochs_no_improve = 0

print("Rozpoczynamy trening!")
print("-" * 30)

for epoch in range(num_epochs):
    
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    loop = tqdm(train_loader, leave=False)
    for (images, tabular), labels in loop:
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images, tabular)
        
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    train_acc = correct_train / total_train

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for (images, tabular), labels in val_loader:
            images, tabular, labels = images.to(device), tabular.to(device), labels.to(device)
            
            outputs = model(images, tabular)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct_val / total_val
    
    scheduler.step(avg_val_loss)
    
    print(f"Epoka {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_melanoma_model.pth")
        print("  -> Nowy najlepszy model zapisany!")
    else:
        epochs_no_improve += 1
        print(f"  -> Brak poprawy od {epochs_no_improve} epok.")
        
        if epochs_no_improve >= patience:
            print("Early Stopping! Przerywam trening, aby uniknąć przeuczenia.")
            break

print("-" * 30)
print("Trening zakończony! Najlepsze wagi znajdują się w pliku 'best_melanoma_model.pth'.")